# Ejercicio: Web Scraping

## Leandro Bravo

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [1]:
from bs4 import BeautifulSoup

file = 'rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [2]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [3]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [4]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [8]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", attrs={"data-tracking-target-url": True})

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    attrs = link['data-tracking-target-url']
    if "recipe/" in attrs:
        recipe_urls.append(attrs)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/
https://www.allrecipes.com/recipe/8998/darn-good-chick

In [12]:
import os
import re
import time
import requests
from urllib.parse import urlparse, urljoin

out_dir = "downloaded_recipes"
os.makedirs(out_dir, exist_ok=True)

links = recipe_urls

base = url if 'url' in globals() else ""

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
})

for i, link in enumerate(links, start=1):
    full = link if urlparse(link).netloc else urljoin(base, link)
    success = False
    for attempt in range(3):
        try:
            resp = session.get(full, timeout=15, allow_redirects=True)
            if resp.status_code == 200:
                path = urlparse(full).path
                name = os.path.basename(path) or f"recipe_{i}"
                name = re.sub(r'[^A-Za-z0-9_.-]+', '_', name)
                if not name.lower().endswith(".html"):
                    name += ".html"
                filename = f"{i:02d}_{name}"
                filepath = os.path.join(out_dir, filename)
                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(resp.text)
                print(f"Saved: {full} -> {filepath}")
                success = True
                break
            if resp.status_code in {402, 403, 429, 500, 502, 503, 504}:
                print(f"Blocked or unavailable: {full} (status {resp.status_code}, attempt {attempt + 1}/3)")
                time.sleep(2)
                continue
            print(f"Unexpected status for {full}: {resp.status_code}")
            break
        except requests.exceptions.RequestException as e:
            print(f"Failed: {full} ({e})")
            break
    if not success:
        fallback_path = os.path.join(out_dir, f"{i:02d}_blocked_{os.path.basename(urlparse(full).path) or i}.txt")
        with open(fallback_path, "w", encoding="utf-8") as f:
            f.write(f"URL: {full}\nStatus: blocked or unavailable\n")
    time.sleep(1)

Saved: https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/ -> downloaded_recipes\01_recipe_1.html
Saved: https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/ -> downloaded_recipes\02_recipe_2.html
Saved: https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/ -> downloaded_recipes\03_recipe_3.html
Saved: https://www.allrecipes.com/recipe/14531/beer-butt-chicken/ -> downloaded_recipes\04_recipe_4.html
Saved: https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/ -> downloaded_recipes\05_recipe_5.html
Saved: https://www.allrecipes.com/recipe/264278/miso-honey-chicken/ -> downloaded_recipes\06_recipe_6.html
Saved: https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/ -> downloaded_recipes\07_recipe_7.html
Saved: https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/ -> downloaded_recipes\08_recipe_8.html
Saved: https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/ -> download

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [15]:
import pandas as pd
# Construcción de corpus
file_path = "downloaded_recipes/*.html"

def extract_recipe_from_html(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        html_content = file.read()
    soup = BeautifulSoup(html_content, "html.parser")

    title_tag = soup.find("meta", {"property": "og:title"})
    if title_tag and title_tag.get("content"):
        title = title_tag["content"]
    else:
        h1 = soup.find("h1")
        title = h1.get_text().strip() if h1 else ""

    desc_tag = soup.find("meta", {"name": "description"})
    description = desc_tag["content"] if desc_tag and desc_tag.get("content") else ""

    ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
    ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section] if ingredients_section else []

    instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
    instructions = [instruction.get_text().strip() for instruction in instructions_section] if instructions_section else []

    nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
    nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section] if nutrition_section else []

    return {
        "title": title,
        "description": description,
        "ingredients": ingredients,
        "instructions": instructions,
        "nutrition_facts": nutrition_facts,
    }

def build_corpus():
    corpus = []
    for filename in os.listdir("downloaded_recipes"):
        if filename.endswith(".html"):
            file_path = os.path.join("downloaded_recipes", filename)
            recipe = extract_recipe_from_html(file_path)
            corpus.append(recipe)
    df_corpus_recipe = pd.DataFrame(corpus)
    return df_corpus_recipe


In [17]:
df_corpus_recipe = build_corpus()

In [21]:
df_corpus_recipe[:1]

,title,description,ingredients,instructions,nutrition_facts
0,Cilantro-Lime Grilled Chicken,This cilantro-lime grilled chicken recipe star...,"[½ cup chopped fresh cilantro, 4 limes, juice...","[Whisk cilantro, lime juice, garlic salt, and ...","[Total Carbohydrate 4g, Dietary Fiber 1g, Tota..."
